# Receipt field extraction: LoRA fine-tuning of a small LM (Avathon Track B / S2)

Runtime: **GPU (T4)**. Run top to bottom. Outputs land in `preds/`, `results/` and `runs/` - commit the small files (`preds/`, `results/`, `runs/*/run_info.json`), not the weights.

Order: setup, data, baselines (no training), train, evaluate, ablations, forgetting check, error analysis, latency, export.

In [ ]:
!nvidia-smi -L

In [ ]:
# TODO: set your repo URL (or upload the repo as a zip, unzip it, and cd into it instead of cloning)
REPO_URL = "https://github.com/pganeshGapps/avathon-slm-doc-extraction.git"
!git clone -q {REPO_URL} repo
%cd repo
!pip -q install -r requirements.txt
!pip -q install -e . --no-deps

In [ ]:
# Downloads SROIE (~500 MB, includes images we drop), builds merchant-grouped train/val and the held-out test split.
!python -m docextract.data --out data/processed
    "!mkdir -p results && cp data/processed/data_card.json results/"

## 1. Baselines (no training)
What the fine-tuned model has to beat: a rule-based extractor, the same base model prompted zero/few-shot, and (optional) a 7B model prompted in 4-bit.

In [ ]:
!python -m docextract.infer --method regex --out preds/regex.jsonl
!python -m docextract.infer --model Qwen/Qwen3-0.6B --shots 0 --out preds/qwen3-0.6b_zeroshot.jsonl
!python -m docextract.infer --model Qwen/Qwen3-0.6B --shots 3 --batch-size 4 --out preds/qwen3-0.6b_3shot.jsonl

In [ ]:
# Optional: the 'just prompt a bigger model' alternative (~5 GB download, slow on a T4).
!python -m docextract.infer --model Qwen/Qwen2.5-7B-Instruct --load-4bit --shots 3 --batch-size 2 --out preds/qwen2.5-7b-instruct_3shot_4bit.jsonl

## 2. Fine-tune (main run)
LoRA r=16, alpha=32, all linear layers, lr 2e-4, 3 epochs. Hyper-parameters are logged to `runs/<name>/train_args.json`, cost to `run_info.json`, loss curves to `log_history.json`.

Run the 16-sample smoke test first if you changed anything.

In [ ]:
!python -m docextract.train --model Qwen/Qwen3-0.6B --out runs/smoke --max-train-samples 16 --epochs 1

In [ ]:
!python -m docextract.train --model Qwen/Qwen3-0.6B --rank 16 --out runs/qwen3-0.6b_lora_r16
!python -m docextract.infer --model Qwen/Qwen3-0.6B --adapter runs/qwen3-0.6b_lora_r16/adapter --out preds/qwen3-0.6b_lora_r16.jsonl

In [ ]:
!python -m docextract.evaluate preds/*.jsonl --out results/summary.md

## 3. Ablations (the evidence behind each design choice)
Each row answers one question in the write-up: LoRA rank, target modules, LoRA vs QLoRA, learning curve (learned vs memorised), base-model choice. Comment out rows you do not have time for.

In [ ]:
# (run name, base model, extra train flags)
ABLATIONS = [
    ("qwen3-0.6b_lora_r4",        "Qwen/Qwen3-0.6B", "--rank 4"),
    ("qwen3-0.6b_lora_r64",       "Qwen/Qwen3-0.6B", "--rank 64"),
    ("qwen3-0.6b_lora_attn_r16",  "Qwen/Qwen3-0.6B", "--rank 16 --targets attn"),
    ("qwen3-0.6b_qlora_r16",      "Qwen/Qwen3-0.6B", "--rank 16 --qlora"),
    ("qwen3-0.6b_lora_r16_n100",  "Qwen/Qwen3-0.6B", "--max-train-samples 100"),
    ("qwen3-0.6b_lora_r16_n300",  "Qwen/Qwen3-0.6B", "--max-train-samples 300"),
    ("qwen3-1.7b_lora_r16",       "Qwen/Qwen3-1.7B", "--batch-size 2 --grad-accum 8 --grad-ckpt"),
    ("qwen2.5-0.5b_lora_r16",     "Qwen/Qwen2.5-0.5B-Instruct", ""),
    ("smollm2-1.7b_lora_r16",     "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--batch-size 2 --grad-accum 8 --grad-ckpt"),
]
for name, model, flags in ABLATIONS:
    !python -m docextract.train --model {model} --out runs/{name} {flags}
    !python -m docextract.infer --model {model} --adapter runs/{name}/adapter --out preds/{name}.jsonl

In [ ]:
!python -m docextract.evaluate preds/*.jsonl --out results/summary.md

## 4. Catastrophic forgetting + latency
Perplexity on WikiText-2 with the adapter off vs on; then single-receipt latency (batch size 1) on 50 receipts.

In [ ]:
!python -m docextract.forgetting --model Qwen/Qwen3-0.6B --adapter runs/qwen3-0.6b_lora_r16/adapter --out results/forgetting_qwen3-0.6b_lora_r16.json
!python -m docextract.infer --model Qwen/Qwen3-0.6B --adapter runs/qwen3-0.6b_lora_r16/adapter --batch-size 1 --limit 50 --out results/latency/qwen3-0.6b_lora_r16_bs1.jsonl
!python -m docextract.evaluate results/latency/qwen3-0.6b_lora_r16_bs1.jsonl

## 5. Qualitative comparison and error analysis

In [ ]:
import json
from docextract.data import load_jsonl
from docextract.metrics import FIELDS, score_example

test = {r["key"]: r for r in load_jsonl("data/processed/test.jsonl")}
base = {r["key"]: r for r in load_jsonl("preds/qwen3-0.6b_3shot.jsonl")}
ft = {r["key"]: r for r in load_jsonl("preds/qwen3-0.6b_lora_r16.jsonl")}

# Same receipts, base (3-shot) vs fine-tuned: pick cases where they differ.
shown = 0
for k, f in ft.items():
    b = base[k]
    if f["pred"] != b["pred"]:
        print(k, "\n  gold :", f["gold"], "\n  base :", b["raw"].strip()[:200], "\n  tuned:", f["raw"].strip()[:200], "\n")
        shown += 1
    if shown == 5:
        break

In [ ]:
# Where does the fine-tuned model fail? Print the receipts it gets wrong, with the failing field.
n_bad = 0
for k, f in ft.items():
    s = score_example(f["pred"], f["gold"])
    bad = [x for x in FIELDS if not s[x]["em"]]
    if bad:
        n_bad += 1
        if n_bad <= 10:
            print(k, "| unseen merchant:", not f["seen_merchant"], "| wrong:", bad)
            for x in bad:
                print(f"   {x}: gold={f['gold'][x]!r} pred={(f['pred'] or {}).get(x)!r} grounded={(f['grounded'] or {}).get(x)}")
print("receipts with >=1 wrong field:", n_bad, "of", len(ft))

## 6. Export for edge deployment (optional)
Merge the adapter into the base model, then convert the merged HF checkpoint with llama.cpp (GGUF, Q4_K_M) or MLX on the Mac. Report size and latency of the quantised model in the write-up.

In [ ]:
!python -m docextract.export --model Qwen/Qwen3-0.6B --adapter runs/qwen3-0.6b_lora_r16/adapter --out merged/qwen3-0.6b_lora_r16
# GGUF (llama.cpp):
# !git clone -q https://github.com/ggml-org/llama.cpp && pip -q install -r llama.cpp/requirements.txt
# !python llama.cpp/convert_hf_to_gguf.py merged/qwen3-0.6b_lora_r16 --outtype q8_0 --outfile merged/qwen3-0.6b_lora_r16.q8_0.gguf
# On the Mac (MLX): mlx_lm.convert --hf-path merged/qwen3-0.6b_lora_r16 -q

## 7. Save artefacts
Freeze the environment (this is the lock file for the README) and download the small outputs to commit.

In [ ]:
!pip freeze | grep -iE "^(torch|transformers|peft|datasets|accelerate|bitsandbytes)==" > results/env_freeze.txt
!python --version >> results/env_freeze.txt
!nvidia-smi -L >> results/env_freeze.txt
!cat results/env_freeze.txt
!zip -qr artifacts.zip preds results runs -x "runs/*/checkpoints/*" "runs/*/adapter/*" "runs/smoke/*"
from google.colab import files
files.download("artifacts.zip")